In [0]:
dbutils.widgets.text("catalog_name", "dbw_lakehouse_dev")
dbutils.widgets.text("bronze_schema", "cloud_practice_bronze")
dbutils.widgets.text("silver_schema", "cloud_practice_silver")

catalog_name = dbutils.widgets.get("catalog_name")
bronze_schema = dbutils.widgets.get("bronze_schema")
silver_schema = dbutils.widgets.get("silver_schema")

bronze_json_table = f"{catalog_name}.{bronze_schema}.orders_json"
silver_orders_table = f"{catalog_name}.{silver_schema}.orders"
silver_customers_table = f"{catalog_name}.{silver_schema}.customers"
silver_order_lines_table = f"{catalog_name}.{silver_schema}.order_lines"

print("Source:", bronze_json_table)
print("Targets:", silver_orders_table, silver_customers_table, silver_order_lines_table)

In [0]:
spark.sql(f"""
CREATE SCHEMA IF NOT EXISTS {catalog_name}.{silver_schema}
COMMENT 'Silver layer - normalized relational tables'
""")

In [0]:
from pyspark.sql.functions import col

bronze_df = (
    spark.table(bronze_json_table)
    .filter(col("order_id").isNotNull())
    .filter(col("customer").isNotNull())
    .filter(col("lines").isNotNull())
)

print("Bronze valid rows:", bronze_df.count())
display(bronze_df.select("order_id", "ordered_at", "status", "customer", "lines").limit(5))

In [0]:
from pyspark.sql.functions import col, explode, current_timestamp

orders_df = bronze_df.select(
    col("order_id"),
    col("ordered_at").cast("timestamp"),
    col("status"),
    col("customer.customer_id").alias("customer_id"),
    current_timestamp().alias("silver_loaded_at"),
)

customers_df = (
    bronze_df.select(
        col("customer.customer_id").alias("customer_id"),
        col("customer.name").alias("name"),
        col("customer.country").alias("country"),
        current_timestamp().alias("silver_loaded_at"),
    )
    .dropDuplicates(["customer_id"])
)

lines_df = (
    bronze_df.select(col("order_id"), explode(col("lines")).alias("line"))
    .select(
        col("order_id"),
        col("line.line_id").alias("line_id"),
        col("line.sku").alias("sku"),
        col("line.qty").cast("int").alias("qty"),
        col("line.unit_price").cast("decimal(10,2)").alias("unit_price"),
        (col("line.qty") * col("line.unit_price")).cast("decimal(12,2)").alias("line_total"),
        current_timestamp().alias("silver_loaded_at"),
    )
)

print("orders:", orders_df.count(), "| customers:", customers_df.count(), "| lines:", lines_df.count())

In [0]:
from delta.tables import DeltaTable

def merge_table(target_table: str, source_df, merge_key: str) -> None:
    if spark.catalog.tableExists(target_table):
        (
            DeltaTable.forName(spark, target_table)
            .alias("t")
            .merge(source_df.alias("s"), f"t.{merge_key} = s.{merge_key}")
            .whenMatchedUpdateAll()
            .whenNotMatchedInsertAll()
            .execute()
        )
    else:
        source_df.write.format("delta").mode("overwrite").saveAsTable(target_table)

merge_table(silver_orders_table, orders_df, "order_id")
merge_table(silver_customers_table, customers_df, "customer_id")
merge_table(silver_order_lines_table, lines_df, "line_id")

print("Silver MERGE completed.")

In [0]:
display(spark.table(silver_orders_table))
display(spark.table(silver_customers_table))
display(spark.table(silver_order_lines_table))

In [0]:
display(
    spark.sql(f"""
        SELECT
          c.country,
          COUNT(DISTINCT o.order_id) AS order_count,
          ROUND(SUM(l.line_total), 2) AS revenue
        FROM {silver_order_lines_table} l
        JOIN {silver_orders_table} o ON l.order_id = o.order_id
        JOIN {silver_customers_table} c ON o.customer_id = c.customer_id
        GROUP BY c.country
        ORDER BY revenue DESC
    """)
)